# Train, Validate, and Predict with a Pretrained `GLACIER` Model

Use this notebook to pull a pretrained `GLACIER` model from Hugging Face, and then get its embeddings to train and evaluate on the [Pgp](https://tdcommons.ai/single_pred_tasks/adme/#pgp-p-glycoprotein-inhibition-broccatelli-et-al) dataset.


In [ ]:
import numpy as np 
import torch
from tqdm import tqdm
from huggingface_hub import snapshot_download
import sys
repo_dir = snapshot_download(repo_id="glacier-hf/GLACIER-100k-MiniMol")
sys.path.append(repo_dir)

# Binary Classification Example: Pgp Dataset

P-glycoprotein Inhibition (Pgp) Inhibition Prediction: Predict whether a compound is a Pgp inhibitor.

Note: This dataset has been preprocessed (as described in the paper) and is ML-ready. 

Broccatelli et al., A Novel Approach for Predicting P-Glycoprotein (ABCB1) Inhibition Using Molecular Interaction Fields. Journal of Medicinal Chemistry, 2011 54 (6), 1740-1751

# Define dataset file paths

In [ ]:
import os 

train_path = os.path.join(".", "example_data", "train_Pgp.tab")
valid_path = os.path.join(".", "example_data", "valid_Pgp.tab")
test_path = os.path.join(".", "example_data", "test_Pgp.tab")

# Load Data

In [ ]:
from data.utils import load_data 

train = load_data(train_path)
valid = load_data(valid_path)
test = load_data(test_path)

# Get SMILES and labels
train_smis = train['smiles'].values
train_labels = train['Y'].values

valid_smis = valid['smiles'].values
valid_labels = valid['Y'].values

test_smis = test['smiles'].values
test_labels = test['Y'].values

# Create dataset and dataloader

In [ ]:
from data.dataloader import SmilesMoleculeDataset, build_dataloader

train_dataset = SmilesMoleculeDataset(train_smis, labels=train_labels)
train_dataloader = build_dataloader(train_dataset, batch_size=32, num_workers=0) 

valid_dataset = SmilesMoleculeDataset(valid_smis, labels=valid_labels)
valid_dataloader = build_dataloader(valid_dataset, batch_size=32, num_workers=0) 

test_dataset = SmilesMoleculeDataset(test_smis, labels=test_labels)
test_dataloader = build_dataloader(test_dataset, batch_size=32, num_workers=0) 

# Get Pretrained `GLACIER` Model from HF

In [ ]:
from glacier_student import Glacier

glacier = Glacier.from_pretrained("glacier-hf/GLACIER-100k-MiniMol")


# Get `GLACIER` Embeddings for Pgp Dataset

In [ ]:
def get_embeddings(dataloader, model):
    embeds_list = []
    labels_list = []
    total_molecules = 0
    with torch.no_grad():
        for batch in tqdm(dataloader):
            batch_on_device = {}
            for k, v in batch.items():
                batch_on_device[k] = v
            
            targets = batch['Y'] 
            current_batch_size = len(targets)
            total_molecules += current_batch_size
            
            embeddings = glacier(batch_on_device) 
            
            embeds_list.append(embeddings.cpu().numpy())
            labels_list.append(np.array(targets))

    # Consolidate results
    X = np.concatenate(embeds_list, axis=0)
    y = np.concatenate(labels_list, axis=0)

    # Format Y 
    if len(y.shape) > 1 and y.shape[1] == 1:
        y = y.ravel()
        
    return X, y

X_train, y_train = get_embeddings(train_dataloader, glacier)
X_valid, y_valid = get_embeddings(valid_dataloader, glacier)
X_test, y_test = get_embeddings(test_dataloader, glacier)

# Define Task-Specific Head

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import roc_auc_score

model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced')
param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}  


# Train & Validate

In [ ]:
# Combine 
X_combined = np.vstack((X_train, X_valid))   
y_combined = np.concatenate((y_train, y_valid)) 

# Make split mask
split_mask = np.concatenate([
    -1 * np.ones(len(X_train)),  # Training rows marked -1 (will be fit)
     0 * np.ones(len(X_valid))   # Validation rows marked 0 (will be evaluated)
])

ps = PredefinedSplit(test_fold=split_mask)


grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1  
)

# Fit
grid_search.fit(X_combined, y_combined)
best_model = grid_search.best_estimator_

print(f"Best Configuration: {grid_search.best_params_}")
print(f"Validation AUROC: {grid_search.best_score_:.4f}\n")

# Eval on Test Set

In [ ]:
# Predict
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

# Score
test_auc = roc_auc_score(y_test, test_probs)

print(f"Test AUROC:  {test_auc:.4f}")
print(f"Done!")
